# 00. 공통 설정 및 환경 확인
설정 우선순위: YAML < 저장한 공통 설정 < 현재 노트북 첫 셀.

## 1. 사용자 설정
기본 5 seed와 세 head입니다. 데이터/완료 checkpoint를 수정하지 않습니다.

In [ ]:
from pathlib import Path
import sys, json
BASE_DIR = Path("F:/FaceX-Zoo")
SETTINGS_PATH = BASE_DIR / "fine-tune/data/notebook_settings.json"
for path in (BASE_DIR, BASE_DIR / "fine-tune"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from modules.notebook_workflow import resolve_settings, save_settings, plan_matrix
# 공통 설정: YAML을 직접 편집할 필요가 없습니다.
SEEDS = [8972, 8973, 8974, 8975, 8976]
HEADS = ["arcface", "adaface", "magface"]
KEEP_RAW_RESULTS = True
NUM_WORKERS = 0  # Windows/Jupyter 안전 모드. 1도 별도 프로세스를 생성합니다.
DEVICE = "cuda:0"
DATASET = dict(name="VGGFace2", raw_root="F:/FaceX-Zoo/VGGFace2",
    train_list="F:/FaceX-Zoo/fine-tune/data/train_list.txt",
    val_list="F:/FaceX-Zoo/fine-tune/data/val_list.txt",
    input_size=[112,112], pixel_mean=127.5, pixel_std=128.0, num_classes=8631)
BACKBONE = dict(type="MobileFaceNet", feat_dim=1024, out_h=7, out_w=7, dropout_rate=0.6)
TRAINING = dict(optimizer="adam", lr=0.001, weight_decay=0.0001, epoches=40,
    milestones=[10,20,30], lr_decay_ratio=0.1, batch_size=64, save_freq=5, print_freq=20)
HEAD_PARAMETERS = {
    "arcface": dict(type="ArcFace", margin_arc=0.6, margin_am=0.0, scale=64.0),
    "adaface": dict(type="AdaFace", m=0.4, h=0.333, scale=64.0, t_alpha=0.01),
    "magface": dict(type="MagFace", margin_am=0.0, scale=64.0, l_a=10.0,
                   u_a=110.0, l_margin=0.45, u_margin=0.8, lamda=20.0),
}
PATHS = dict(checkpoint_dir="F:/FaceX-Zoo/fine-tune/checkpoints",
             export_dir="F:/FaceX-Zoo/fine-tune/checkpoints/exported",
             report_dir="F:/FaceX-Zoo/fine-tune/reports/campaigns")
REUSE_COMPLETED = True
# 현재 노트북 변수 -> 실제 모듈 설정. 입력 112x112 이외는 현 구조에서 명시적 중단.
DATASET.update(train_archive=str(Path(DATASET["raw_root"])/"data/vggface2_train.tar.gz"),
               test_archive=str(Path(DATASET["raw_root"])/"data/vggface2_test.tar.gz"),
               meta_csv=str(Path(DATASET["raw_root"])/"meta/identity_meta.csv"))
settings = resolve_settings(BASE_DIR, SETTINGS_PATH, overrides={
    "project": {"seed": SEEDS[0], "seed_list": SEEDS, "keep_raw_results": KEEP_RAW_RESULTS, "device": DEVICE},
    "campaign": {"heads": HEADS, "reuse_completed": REUSE_COMPLETED},
    "dataset": DATASET, "backbone": BACKBONE, "heads": HEAD_PARAMETERS,
    "training": {**TRAINING, "num_workers": NUM_WORKERS}, "paths": PATHS,
})


## 2. 설정 저장

In [ ]:
print(save_settings(SETTINGS_PATH, settings))

## 3. 실제 환경 확인

In [ ]:
from modules.notebook_workflow import inspect_environment
environment = inspect_environment(BASE_DIR, settings)
print(json.dumps(environment, ensure_ascii=False, indent=2))

## 4. 실행 계획

In [ ]:
for stage in ("pretrain", "finetune", "export"):
    plan = plan_matrix(settings, stage)
    print(stage, len(plan), [r["run_id"] for r in plan])

## 5. 확인 범위

In [ ]:
print("부분 환경 점검입니다. 압축 전체 무결성·전체 이미지 decode·학습·FIQA 평가는 미실행입니다.")
print("안내:", BASE_DIR / "fine-tune/README.md")